## Zad 1.
Zaimplementuj funkcję `trace_of_jacobian(z, f)`, która przyjmuje:
- `z`: tensor o kształcie `(batch_size, d)` (punkt, w którym liczymy Jakobian),
- `f`: funkcję, która dla danego `z` zwraca tensor o kształcie `(batch_size, d)` (wartość pola wektorowego)

Oblicza ślad Jakobianu $\text{tr}\left( \frac{\partial f}{\partial z} \right)$ dla każdego elementu w batchu, używając estymator Hutchinson:
$$
\text{tr}(A) = \mathbb{E}_{v \sim \mathcal{N}(0,I)} \left[ v^T A v \right]
$$
gdzie $A = \frac{\partial f}{\partial z}$. Użyj `torch.autograd.grad`, `.backward()` lub `torch.autograd.functional.jvp` z odpowiednim rozdzieleniem gradientów.

In [ ]:
import torch
import torch.nn as nn

def trace_of_jacobian(z, f):
    """
    Oblicza tr(Jacobian of f wrt z) dla każdego elementu w batchu.
    
    Args:
        z: Tensor shape (batch_size, d), wymaga gradientu.
        f: Funkcja, która bierze z i zwraca tensor tego samego kształtu.
    
    Returns:
        trace: Tensor shape (batch_size,)
    """
    pass




batch_size = 4
d = 3
z = torch.randn(batch_size, d, requires_grad=True)

class SimpleF(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 10),
            nn.Tanh(),
            nn.Linear(10, d)
        )
    def forward(self, z):
        return self.net(z)

f_model = SimpleF()

def f_func(z):
    return f_model(z)

trace = trace_of_jacobian(z, f_func)
print("Trace shape:", trace.shape)
print("Trace values:", trace)

## Zad 2.
Zaimplementuj funkcję `divergence_via_hutchinson(f, z, n_samples=1)`, która oblicza **dywergencję** (ślad Jakobianu) pola wektorowego $f: \mathbb{R}^d \to \mathbb{R}^d$ w punkcie $z$ używając **estymatora Hutchinsona** z `n_samples` próbek.

Dywergencja pola wektorowego $f$ w punkcie $z$ definiowana jest jako:
$$
\text{div} f(z) = \text{tr}\left( \frac{\partial f}{\partial z} \right) = \sum_{i=1}^d \frac{\partial f_i}{\partial z_i}
$$

Estymator Hutchinsona:
$$
\text{tr}(J_f(z)) = \mathbb{E}_{v \sim p(v)} [v^T J_f(z) v] \approx \frac{1}{\text{n\_samples}} \sum_{k=1}^{\text{n\_samples}} v_k^T J_f(z) v_k
$$
gdzie $J_f(z) = \frac{\partial f}{\partial z}$ to Jakobian pola $f$ w punkcie $z$.

**Wymagania:**
- Funkcja powinna obsługiwać batch (tensor `z` o kształcie `(batch_size, d)`)
- Użyj rozkładu normalnego $v \sim \mathcal{N}(0, I)$ dla wektorów próbkujących
- Zaimplementuj efektywne obliczenie $v^T J_f(z) v$ za pomocą wywołania backward na próbce.

## Zad 3.
Dane jest pole wektorowe zależne od czasu $f(z, t) = A(t) \cdot \sigma(W(t) z + b(t))$, gdzie:
- $A(t) \in \mathbb{R}^{d \times d}$, $W(t) \in \mathbb{R}^{d \times d}$, $b(t) \in \mathbb{R}^{d}$ są funkcjami czasu
- $\sigma$ to funkcja aktywacji (tanh)
- $z \in \mathbb{R}^d$ to stan

Zaimplementuj symulację, która weryfikuje **równanie ciągłości** (twierdzenie Liouville'a) dla dynamiki ODE:
$$
\frac{dz(t)}{dt} = f(z(t), t)
$$
$$
\frac{d \log p(z(t))}{dt} = -\text{div} f(z(t), t) = -\text{tr}\left( \frac{\partial f}{\partial z} \right)
$$

**Kroki:**
1. Zdefiniuj pole wektorowe $f(z, t)$ jako sieć neuronową zależną od czasu
2. Całkuj ODE od $t_0$ do $t_1$ używając prostego schematu Eulera
3. Na każdym kroku czasowym obliczaj dywergencję pola za pomocą funkcji z poprzedniego zadania
4. Porównaj dwie metody obliczenia zmiany gęstości:
   - **Metoda 1:** Całkowanie dywergencji: $\log p(z(t_1)) - \log p(z(t_0)) = -\int_{t_0}^{t_1} \text{div} f(z(\tau), \tau) d\tau$
   - **Metoda 2:** Bezpośrednie obliczenie przez zmianę zmiennych


## Zad 4.
Korzystając z kodów z [repozytorium GitHub](https://github.com/tatsy/normalizing-flows-pytorch), naucz model `FFJORD` na zbiorze mnist. Po nauczeniu modelu wygeneruj kilka obrazków z przestrzeni latent. Użyj funkcji [t-sne](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) do wizualizacji przestrzeni latentej na podstawie punktów reprezentujących dane (obrazy), stosując różne kolory dla poszczególnych klas.  